# 2110446 DATA SCIENCE AND DATA ENGINEERING

## **Unit 08:** Spark

- **Author:** Worralop Srichainont
- **Year:** 2025 (Semester 2)

## **Homework:** Spark

With file `netflix-rotten-tomatoes-metacritic-imdb.csv` in git repo, use the spark assignment notebook in the git repo and provide the following answers:

1. Answer questions in the question set
2. Save your notebook solution in the PDF format  (used File -> Print Preview and then export as PDF or print to PDF with your browser).

# Dependencies

In [71]:
%pip install pyspark

In [72]:
from pyspark import SparkFiles
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, desc, explode, split, trim

# Session Initialization

In [73]:
APP_NAME = "DSDE_HW8"

In [74]:
spark = SparkSession.builder.appName(APP_NAME).master("local[*]").getOrCreate()

# Load Data

In [75]:
DATA_URL = "https://raw.githubusercontent.com/pvateekul/2110446_DSDE_2025s2/refs/heads/main/code/Week11_Spark/netflix-rotten-tomatoes-metacritic-imdb.csv"
DATA_PATH = "netflix-rotten-tomatoes-metacritic-imdb.csv"

In [76]:
spark.sparkContext.addFile(DATA_URL)
df = spark.read.csv(SparkFiles.get(DATA_PATH), header=True, inferSchema=True)

Get brief details of the data

In [77]:
n_rows = df.count()
n_cols = len(df.columns)
print(f"There are {n_rows} rows and {n_cols} columns")

There are 15480 rows and 29 columns


Visualize data schema

In [78]:
df.printSchema()

root
 |-- Title: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Languages: string (nullable = true)
 |-- Series or Movie: string (nullable = true)
 |-- Hidden Gem Score: double (nullable = true)
 |-- Country Availability: string (nullable = true)
 |-- Runtime: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Writer: string (nullable = true)
 |-- Actors: string (nullable = true)
 |-- View Rating: string (nullable = true)
 |-- IMDb Score: string (nullable = true)
 |-- Rotten Tomatoes Score: string (nullable = true)
 |-- Metacritic Score: string (nullable = true)
 |-- Awards Received: double (nullable = true)
 |-- Awards Nominated For: double (nullable = true)
 |-- Boxoffice: string (nullable = true)
 |-- Release Date: string (nullable = true)
 |-- Netflix Release Date: string (nullable = true)
 |-- Production House: string (nullable = true)
 |-- Netflix Link: string (nullable = true)
 |-- IMDb Link: string (null

Visualize data samples

In [79]:
df.show(10)

+-------------------+--------------------+--------------------+--------------------+---------------+----------------+--------------------+------------+---------------+--------------------+--------------------+-----------+----------+---------------------+----------------+---------------+--------------------+----------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------------------+--------------------+------------+
|              Title|               Genre|                Tags|           Languages|Series or Movie|Hidden Gem Score|Country Availability|     Runtime|       Director|              Writer|              Actors|View Rating|IMDb Score|Rotten Tomatoes Score|Metacritic Score|Awards Received|Awards Nominated For| Boxoffice|Release Date|Netflix Release Date|    Production House|        Netflix Link|           IMDb Link|             Summary|IMDb Votes|               Image|      

# **Problem 1**: Hidden Gem Score

What is the maximum and average of the overall hidden gem score?

In [80]:
COL_NAME = "Hidden Gem Score"

In [81]:
df.describe(COL_NAME).show()

+-------+------------------+
|summary|  Hidden Gem Score|
+-------+------------------+
|  count|             13379|
|   mean| 5.937551386501234|
| stddev|2.2502018012760523|
|    min|               0.6|
|    max|               9.8|
+-------+------------------+



# **Problem 2**: Korean Movies

How many movies and series that are available in Korean language?

In [82]:
COL_NAME = "Languages"
KEYWORD = "Korean"

In [83]:
df_korean = df.filter(col(COL_NAME).contains(KEYWORD))
print(f"There are {df_korean.count()} movies and series available in Korean")

There are 735 movies and series available in Korean


# **Problem 3**: Best Director

Which director has the highest average hidden gem score?

In [84]:
DIRECTOR_COL_NAME = "Director"
SCORE_COL_NAME = "Hidden Gem Score"
AVG_SCORE_COL_NAME = "avg_score"

In [85]:
# Remove rows with null value in director column
df_cleaned = df.filter(col(DIRECTOR_COL_NAME).isNotNull())

# Group rows with the same director and sort the average score descendingly
df_director_avg_score = (
    df_cleaned.groupBy(DIRECTOR_COL_NAME)
    .agg(avg(SCORE_COL_NAME).alias(AVG_SCORE_COL_NAME))
    .orderBy(desc(AVG_SCORE_COL_NAME))
)

In [86]:
df_director_avg_score.limit(1).show()

+-----------+---------+
|   Director|avg_score|
+-----------+---------+
|Dorin Marcu|      9.8|
+-----------+---------+



# **Problem 4**: Movie Genre

How many genres are there in the data set?

In [87]:
GENRE_COL_NAME = "Genre"
SINGLE_GENRE_COL_NAME = "single_genre"

In [88]:
# Remove rows with null value in genre column
df_cleaned = df.filter(col(GENRE_COL_NAME).isNotNull())

# Split genre column into multiple rows
df_genre = (
    df_cleaned.withColumn(
        SINGLE_GENRE_COL_NAME, explode(split(col(GENRE_COL_NAME), ","))
    )
    .withColumn(SINGLE_GENRE_COL_NAME, trim(col(SINGLE_GENRE_COL_NAME)))
    .groupBy(SINGLE_GENRE_COL_NAME)
    .count()
    .orderBy(desc("count"))
)

In [ ]:
n_genre = df_genre.count()
print(f"There are total {n_genre} movie genres")

In [ ]:
df_genre.show(n=n_genre)